# Core ranking: PageRank and the in-degree comparison

## Purpose

This notebook discharges requirements 2, 3 and 4 of the brief on the
real CAIDA AS graph: PageRank with dangling-node treatment, an
interpretation of the top-ranked ASes, and the comparison against
in-degree with the disagreements explained.

The graph is the *provisional primary* variant built in notebook 01:
**peering `both`, vote `customer_to_provider`, IXPs kept** (80,510
nodes, 884,790 edges, 0 dangling). Two modelling choices stay open and
are reported here as sensitivities rather than decided: the vote
direction (a customer paying for transit endorses its provider, so the
edge runs customer to provider; notebook 01 has the counts) and whether
peer-to-peer links contribute edges (peering `both` adds an edge each
way). Excluding the 28 header-declared IXP ASes was examined in
notebook 01 and is inert on this snapshot, since none of them appears in
any relationship line, so it is not revisited here.

The computation lives in `pagerank`; this notebook loads, runs, tabulates
and plots.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pagerank import (
    from_edge_list,
    pagerank_power,
    ranking,
    rank_positions,
    top_k_overlap,
    kendall_tau,
)
from pagerank.caida import build_edge_list, load_build, read_as_rel

REPO = Path.cwd().parent
RAW = REPO / 'data' / 'raw'
PROCESSED = REPO / 'data' / 'processed'
FIGURES = REPO / 'results' / 'figures'
TABLES = REPO / 'results' / 'tables'
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

SNAPSHOT = '20260901'
AS_REL = RAW / f'{SNAPSHOT}.as-rel.bz2'
PRIMARY_NPZ = PROCESSED / f'{SNAPSHOT}_peering-both_vote-c2p_ixp-kept.npz'
NONE_NPZ = PROCESSED / f'{SNAPSHOT}_peering-none_vote-c2p_ixp-kept.npz'

ALPHA = 0.85
TOL = 1e-12
t_start = time.perf_counter()

## 1. Loading the graph

`load_build` reads the saved primary build; `from_edge_list` turns its
edge array into the sparse transition structure. The edge array is passed
straight in (the array fast path), so the 884k-edge build loads in well
under a second. `asns` is kept to label nodes by ASN. The header clique
of 18 transit-free ASes is read from the raw snapshot for the sanity
check below.

The print also reports the share of edges that come from peer-to-peer
rows. Peering is symmetric, so each of the 358,942 p2p relationships adds
two edges, and these make up about 81% of all 884,790 edges. That
majority is worth keeping in mind throughout: under peering `both` the
graph is dominated by peering links, so a network with many peers both
collects votes and, having a large out-degree, hands out small ones.

In [ ]:
build = load_build(PRIMARY_NPZ)
asns = build.asns

t0 = time.perf_counter()
graph = from_edge_list(build.edges, build.n)
load_seconds = time.perf_counter() - t0

# The clique and the customer/peer facts come from the raw snapshot;
# re-reading it takes about 0.4 s and keeps the edge counts honest.
snap = read_as_rel(AS_REL)
clique_asns = np.array(snap.clique, dtype=np.int64)

# Share of edges that are peering edges. Peering is symmetric, so each
# p2p relationship contributes two edges; c2p contributes one each.
peering_edge_share = 2 * build.n_p2p_used / graph.n_edges

print(f'peering={build.peering}, vote={build.vote}, excluded={len(build.excluded_asns)}')
print(f'nodes {graph.n}, edges {graph.n_edges}, dangling {int(graph.dangling.sum())}')
print(f'peering-edge share (2 * n_p2p / n_edges): {peering_edge_share:.3f}')
print(f'from_edge_list on the edge array: {load_seconds*1000:.0f} ms')
assert graph.n == 80510
assert graph.n_edges == 884790
assert int(graph.dangling.sum()) == 0
assert len(clique_asns) == 18

## 2. PageRank on the AS graph

Power iteration at damping `alpha = 0.85` to an L1 residual below
`1e-12`. The primary graph has no dangling node, so the dangling term is
inactive here (it is exercised in sensitivity 1 below); correctness of
the treatment itself is proved on the toy graphs in notebook 00. The
vector sums to 1 by construction; the print confirms the drift is at
floating-point level.

In [ ]:
result = pagerank_power(graph, alpha=ALPHA, tol=TOL)
pr = result.vector

print(f'iterations     {result.iterations}')
print(f'converged      {result.converged}')
print(f'final residual {result.residuals[-1]:.2e}')
print(f'vector.sum()   {pr.sum():.15f}')
assert result.converged
assert abs(pr.sum() - 1.0) < 1e-12

# Recorded for the convergence study (notebook 05): power iteration at
# alpha=0.85 reaches tol 1e-12 on this graph in the printed iteration
# count, comfortably under 200.
pr_iterations_primary = result.iterations

## 3. The top of the ranking

The 20 highest-PageRank ASes, each with its in-degree and in-degree rank,
out-degree, number of customers and peers (from the raw snapshot), and a
flag for clique membership. Customers are the `as2` of a customer-provider
(`-1`) row; peers are either endpoint of a peer-to-peer (`0`) row. Then
the **clique sanity check**: where each of the 18 transit-free ASes lands
by PageRank, and how many sit inside the top 100. If any falls outside,
that is reported as a finding, not adjusted away.

In [ ]:
# Per-ASN customer and peer counts from the raw snapshot.
rel = snap.relationships
c2p = rel[rel[:, 2] == -1]
p2p = rel[rel[:, 2] == 0]

# A provider (as1 of a -1 row) gains one customer per such row.
n_customers_by_asn = pd.Series(c2p[:, 0]).value_counts()
# A peer appears as either endpoint of a 0 row.
peer_endpoints = np.concatenate([p2p[:, 0], p2p[:, 1]])
n_peers_by_asn = pd.Series(peer_endpoints).value_counts()

in_deg = graph.in_degree
out_deg = graph.out_degree
in_rank = rank_positions(in_deg.astype(np.float64))
pr_rank = rank_positions(pr)
clique_set = set(int(a) for a in clique_asns)

order_pr = ranking(pr)


def row_for(node):
    asn = int(asns[node])
    return {
        'rank': int(pr_rank[node]),
        'asn': asn,
        'pagerank': float(pr[node]),
        'in_degree': int(in_deg[node]),
        'in_degree_rank': int(in_rank[node]),
        'out_degree': int(out_deg[node]),
        'customers': int(n_customers_by_asn.get(asn, 0)),
        'peers': int(n_peers_by_asn.get(asn, 0)),
        'clique': asn in clique_set,
    }


top20 = pd.DataFrame([row_for(int(n)) for n in order_pr[:20]])
top20_path = TABLES / '02_top20_pagerank.csv'
top20.to_csv(top20_path, index=False)
print('wrote', top20_path.relative_to(REPO))
top20

In [ ]:
# Clique sanity check: PageRank rank of each of the 18 clique ASes.
clique_nodes = np.array(
    [int(np.searchsorted(asns, a)) for a in clique_asns], dtype=np.int64
)
assert np.array_equal(asns[clique_nodes], clique_asns)

clique_tbl = pd.DataFrame({
    'asn': clique_asns,
    'pagerank_rank': pr_rank[clique_nodes],
    'in_degree_rank': in_rank[clique_nodes],
}).sort_values('pagerank_rank').reset_index(drop=True)

in_top_100 = int((clique_tbl['pagerank_rank'] <= 100).sum())
worst = clique_tbl.iloc[-1]
print(f'clique members in the PageRank top 100: {in_top_100} of 18')
print(f'worst-placed clique AS: {int(worst.asn)} at PageRank rank {int(worst.pagerank_rank)}')
clique_tbl

## 4. Distribution of PageRank

Two views of how PageRank mass is spread across the 80k ASes: the
complementary cumulative distribution (CCDF) on log-log axes, the same
helper notebook 01 used for degree, and the brief's histogram on a log
x-axis. Both show the heavy tail: a handful of ASes hold orders of
magnitude more score than the bulk, which is the structural reason
PageRank and in-degree agree at the very top and part ways in the tail.

In [ ]:
def ccdf(values):
    values = np.sort(values)
    n = values.size
    # P(X >= x) at each distinct value.
    unique, idx = np.unique(values, return_index=True)
    survival = 1.0 - idx / n
    return unique, survival


fig, axes = plt.subplots(1, 2, figsize=(11, 4))

x, y = ccdf(pr)
axes[0].loglog(x, y, marker='.', linestyle='none', markersize=3)
axes[0].set_xlabel('PageRank score p')
axes[0].set_ylabel('P(PageRank >= p)')
axes[0].set_title('PageRank CCDF (log-log)')
axes[0].grid(True, which='both', alpha=0.3)

axes[1].hist(pr, bins=np.logspace(np.log10(pr.min()), np.log10(pr.max()), 50))
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('PageRank score (log)')
axes[1].set_ylabel('number of ASes')
axes[1].set_title('PageRank histogram (log x)')
axes[1].grid(True, which='both', alpha=0.3)

fig.suptitle('Distribution of PageRank (peering both, IXPs kept)')
fig.tight_layout()
dist_path = FIGURES / '02_pagerank_distribution.png'
fig.savefig(dist_path, dpi=150)
print('wrote', dist_path.relative_to(REPO))
plt.show()

## 5. PageRank versus in-degree

Requirement 4. How much do the two orderings agree? Top-k overlap at
10/50/100, Kendall's tau-b over all nodes, and tau-b over the union of
the top 1000 by either measure (the tail of mutual ties drags the
all-node tau, so the top-1000 figure is the more telling one). The
scatter plots PageRank against in-degree on log-log axes for every AS
with in-degree at least 1, clique members highlighted, and the top 5 by
PageRank annotated with their ASN.

In [ ]:
union_top = np.union1d(ranking(pr)[:1000], ranking(in_deg.astype(np.float64))[:1000])

summary = pd.DataFrame([
    {'metric': 'overlap@10', 'value': top_k_overlap(pr, in_deg.astype(np.float64), 10)},
    {'metric': 'overlap@50', 'value': top_k_overlap(pr, in_deg.astype(np.float64), 50)},
    {'metric': 'overlap@100', 'value': top_k_overlap(pr, in_deg.astype(np.float64), 100)},
    {'metric': 'tau_b_all', 'value': kendall_tau(pr, in_deg.astype(np.float64))},
    {'metric': 'tau_b_top1000_union',
     'value': kendall_tau(pr, in_deg.astype(np.float64), subset=union_top)},
])
summary_path = TABLES / '02_pagerank_vs_indegree_summary.csv'
summary.to_csv(summary_path, index=False)
print('wrote', summary_path.relative_to(REPO))
summary

In [ ]:
mask = in_deg >= 1
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(in_deg[mask], pr[mask], s=4, alpha=0.25, label='AS (in-degree >= 1)')

clique_mask = np.zeros(graph.n, dtype=bool)
clique_mask[clique_nodes] = True
cm = clique_mask & mask
ax.scatter(in_deg[cm], pr[cm], s=28, color='crimson', label='clique')

# Annotate only the top 5 by PageRank; labelling the top 10 by either
# measure crowded the top-right corner into illegibility. A small fixed
# offset keeps each label clear of its marker.
for node in ranking(pr)[:5]:
    ax.annotate(str(int(asns[node])), (in_deg[node], pr[node]),
                fontsize=8, xytext=(5, 5), textcoords='offset points')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('in-degree')
ax.set_ylabel('PageRank')
ax.set_title('PageRank vs in-degree (peering both, IXPs kept)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
scatter_path = FIGURES / '02_pagerank_vs_indegree.png'
fig.savefig(scatter_path, dpi=150)
print('wrote', scatter_path.relative_to(REPO))
plt.show()

## 6. Where they disagree

The ASes the two measures most disagree about, restricted to the union of
the top 1000 by either (so the comparison is among networks that matter
to at least one measure). We rank these by `|PageRank rank - in-degree
rank|` and split them into those PageRank promotes (better PageRank rank
than in-degree rank) and those it demotes, taking the 10 largest shifts
each way. Ranking over the whole graph instead would return only the
promoted direction, since the biggest shifts there are obscure ASes
PageRank lifts from the tail.

To find the mechanism, we compute three quantities over each AS's
in-neighbours (the networks pointing at it) before writing any prose:

- `nbr_pr_rel`: the mean PageRank of the in-neighbours, divided by the
  graph-wide mean PageRank. How important, on average, the pointers are.
- `nbr_outdeg_median`: the median out-degree of the in-neighbours. How
  many ways each pointer splits its vote.
- `contrib_per_link`: the mean of `pr[tail] / out_degree[tail]` over the
  in-neighbours. This is exactly the per-link term of the PageRank update
  `x[head] += alpha * pr[tail] / out_degree[tail]`, so it is what each
  in-link actually delivers, not an interpretation laid on top.

In [ ]:
# For each node, summarise its in-neighbours (the tails pointing at it).
# transpose[head, tail] != 0 for an edge tail -> head, so row `node` of
# the stored transpose holds node's in-neighbours: the nonzero columns of
# that row are the tails. We report the mean importance of those tails,
# their median out-degree, and the mean per-link contribution they
# deliver, which is the term `pr[tail] / out_degree[tail]` from the update
# rule.
pt_csr = graph.transpose.tocsr()
mean_pr = float(pr.mean())


def in_neighbour_stats(node):
    start, end = pt_csr.indptr[node], pt_csr.indptr[node + 1]
    tails = pt_csr.indices[start:end]
    if tails.size == 0:
        return np.nan, np.nan, np.nan
    nbr_pr_rel = float(pr[tails].mean() / mean_pr)
    nbr_outdeg_median = float(np.median(out_deg[tails]))
    contrib_per_link = float((pr[tails] / out_deg[tails]).mean())
    return nbr_pr_rel, nbr_outdeg_median, contrib_per_link


# largest shift each way *within* the union of the two top 1000s. The
# signed shift is negative when PageRank ranks the node better (smaller
# rank number) than in-degree does.
union_nodes = np.array(sorted(int(n) for n in union_top), dtype=np.int64)
signed_shift = pr_rank.astype(np.int64) - in_rank.astype(np.int64)


def rows_for_direction(candidate_nodes, direction, k=10):
    # Sort by descending absolute shift, ties by ascending index.
    ordered = sorted(candidate_nodes, key=lambda n: (-abs(int(signed_shift[n])), int(n)))
    out = []
    for node in ordered[:k]:
        nbr_pr_rel, nbr_outdeg_median, contrib_per_link = in_neighbour_stats(node)
        rec = row_for(node)
        rec['in_degree_rank'] = int(in_rank[node])
        rec['shift'] = abs(int(signed_shift[node]))
        rec['nbr_pr_rel'] = round(nbr_pr_rel, 1)
        rec['nbr_outdeg_median'] = round(nbr_outdeg_median, 1)
        rec['contrib_per_link'] = contrib_per_link
        rec['direction'] = direction
        out.append(rec)
    return out


promoted_nodes = [int(n) for n in union_nodes if signed_shift[n] < 0]
demoted_nodes = [int(n) for n in union_nodes if signed_shift[n] > 0]
promoted = rows_for_direction(promoted_nodes, 'pagerank_higher')
demoted = rows_for_direction(demoted_nodes, 'indegree_higher')

promoted_df = pd.DataFrame(promoted[:10])
demoted_df = pd.DataFrame(demoted[:10])
disagreements = pd.concat([promoted_df, demoted_df], ignore_index=True)
disagreements_path = TABLES / '02_disagreements.csv'
disagreements.to_csv(disagreements_path, index=False)
print('wrote', disagreements_path.relative_to(REPO))

for label, frame in (('pagerank_higher', promoted_df), ('indegree_higher', demoted_df)):
    print(
        f'{label:16s} median: nbr_pr_rel={frame["nbr_pr_rel"].median():.1f}, '
        f'nbr_outdeg_median={frame["nbr_outdeg_median"].median():.0f}, '
        f'contrib_per_link={frame["contrib_per_link"].median():.2e}'
    )
disagreements

The mechanism is in the three computed columns, and it is not the one a
first guess ("PageRank promotes ASes pointed at by more important
networks") would suggest. The pointing networks are about **equally
important** in both directions: `nbr_pr_rel` is roughly 130 for the
promoted ASes and 110 for the demoted ones, both far above the graph mean
either way. What differs, by a factor of about twenty, is
`contrib_per_link`, and the reason is `nbr_outdeg_median`: the ASes
in-degree ranks higher collect a hundred or more in-links from peering
hubs whose out-degree is a thousand or more, so the update rule divides
each of those votes by a thousand; the ASes PageRank promotes receive a
handful of links from networks with a few dozen out-links each, so every
link delivers roughly twenty times as much. This is vote dilution by the
pointer's out-degree, the `pr[tail] / out_degree[tail]` term of the
update itself, not a difference in how important the pointers are. The
business reading of why these particular networks sit where they do is
the author's; the arithmetic above is the fact.

## 7. Dangling nodes on the primary graph

The primary graph has **zero** dangling nodes: with peering `both`, every
provider-free AS still has outgoing peering edges, so no node leaks mass.
The dangling-node treatment is therefore correct (proved on the toy graph
in notebook 00) but inactive here. The sensitivity below drops peering,
which makes the transit-free core dangle and shows the treatment at work.

In [ ]:
print(f'dangling nodes on the primary graph: {int(graph.dangling.sum())}')

## 8. Sensitivity 1 - peering `none`

Dropping peer-to-peer edges leaves only customer-to-provider edges. The
494 provider-free ASes (all 18 clique members among them, per notebook
01) now have no out-edge and dangle, so the dangling treatment becomes
active. We report the iteration count, the total PageRank mass sitting on
the 494 dangling nodes, where the clique lands, and the overlap and tau
against the primary ranking.

In [ ]:
none_build = load_build(NONE_NPZ)
assert np.array_equal(none_build.asns, asns)  # same node indexing
none_graph = from_edge_list(none_build.edges, none_build.n)
none_result = pagerank_power(none_graph, alpha=ALPHA, tol=TOL)
pr_none = none_result.vector

n_dangling_none = int(none_graph.dangling.sum())
dangling_mass = float(pr_none[none_graph.dangling].sum())
none_pr_rank = rank_positions(pr_none)

none_summary = pd.DataFrame([
    {'metric': 'iterations', 'value': none_result.iterations},
    {'metric': 'dangling_nodes', 'value': n_dangling_none},
    {'metric': 'dangling_pagerank_mass', 'value': dangling_mass},
    {'metric': 'overlap@10', 'value': top_k_overlap(pr, pr_none, 10)},
    {'metric': 'overlap@50', 'value': top_k_overlap(pr, pr_none, 50)},
    {'metric': 'overlap@100', 'value': top_k_overlap(pr, pr_none, 100)},
    {'metric': 'tau_b_all', 'value': kendall_tau(pr, pr_none)},
])
none_summary_path = TABLES / '02_sensitivity_peering_none.csv'
none_summary.to_csv(none_summary_path, index=False)
print('wrote', none_summary_path.relative_to(REPO))

clique_none = pd.DataFrame({
    'asn': clique_asns,
    'rank_primary': pr_rank[clique_nodes],
    'rank_peering_none': none_pr_rank[clique_nodes],
}).sort_values('rank_peering_none').reset_index(drop=True)
print(f'dangling PageRank mass under peering none: {dangling_mass:.4f}')
none_summary

In [ ]:
clique_none

Peering `none` makes precisely the most central ASes dangle, so a
noticeable share of the total mass ends up redistributed through the
teleport term rather than flowing along edges. The clique ranks move,
and the overlap against the primary quantifies how much the ranking
depends on the peering choice. No decision is taken here; this is a
sensitivity for the author's modelling write-up.

## 9. Sensitivity 2 - vote reversed

The vote direction is the other open choice. Reversing it
(`provider_to_customer`, peering both) sends mass down the hierarchy
towards customers instead of up towards providers. We build this variant
from the snapshot in the notebook, run PageRank, and compare its top 10
and its tau-b against the primary.

In [ ]:
rev_build = build_edge_list(snap, peering='both', vote='provider_to_customer')
assert np.array_equal(rev_build.asns, asns)
rev_graph = from_edge_list(rev_build.edges, rev_build.n)
rev_result = pagerank_power(rev_graph, alpha=ALPHA, tol=TOL)
pr_rev = rev_result.vector
rev_pr_rank = rank_positions(pr_rev)

rev_top10 = pd.DataFrame([
    {
        'rank': int(rev_pr_rank[int(n)]),
        'asn': int(asns[int(n)]),
        'pagerank_reversed': float(pr_rev[int(n)]),
        'rank_primary': int(pr_rank[int(n)]),
        'clique': int(asns[int(n)]) in clique_set,
    }
    for n in ranking(pr_rev)[:10]
])
tau_rev = kendall_tau(pr, pr_rev)
rev_top10['tau_b_vs_primary'] = tau_rev
rev_path = TABLES / '02_sensitivity_vote_reversed.csv'
rev_top10.to_csv(rev_path, index=False)
print('wrote', rev_path.relative_to(REPO))
print(f'tau_b (reversed vote vs primary): {tau_rev:.4f}')
rev_top10

Reversing the vote produces a very different ranking: mass flows to the
ASes with the most customers rather than the most providers, and the
tau-b against the primary measures how far apart the two orderings are.
This confirms the vote direction is a consequential modelling decision,
not a cosmetic one. The choice stays with the author.

## 10. Interpretation (draft)

> Draft of the factual observations. The interpretation in the submitted
> version is written by the author.

- Under peering `both` the top of the ranking is dominated by **heavily
  peered networks**, not by the largest transit providers: 11 of the top
  20 have fewer than 100 customers but thousands of peers (AS24482: 59
  customers, 9,779 peers; AS1000: 3 and 5,138; AS32590: 0 and 5,159). The
  large transit providers in the top 20 are the four clique members
  AS3356 and AS174 (about 6,470 customers each), AS1299 and AS6461, and
  they reach the top on customers rather than peers.
- 81% of the edges are peering edges (2 x 358,942 of 884,790), so under
  peering `both` a network's peer count largely determines where it sits:
  many peers mean many in-links, which lift it, even though the same large
  out-degree means each vote it casts is small. The ranking is as much
  about peering connectivity as about the transit hierarchy, which is the
  central fact for the peering modelling decision.
- PageRank and in-degree agree strongly at the very top (overlap at 10
  and 50 is high) and diverge further down, where the tau over all nodes
  is dragged by the long tail of tied low-degree ASes.
- Where they disagree, the mechanism is vote dilution: the pointing
  networks are equally important in both directions (`nbr_pr_rel` about
  130 vs 110), but the ASes in-degree ranks higher are pointed at by
  high-out-degree hubs, so each in-link contributes about twenty times
  less (`contrib_per_link` 1.9e-5 vs 8.9e-7). It is the
  `pr[tail] / out_degree[tail]` term of the update, not pointer
  importance.
- The primary graph has no dangling node, so the dangling treatment is
  correct and inactive; dropping peering makes the 494 provider-free ASes
  dangle and activates it, redistributing the printed mass.
- Both open modelling choices matter: peering `none` moves the clique and
  lowers the overlap against the primary, and reversing the vote produces
  a sharply different ranking (low tau-b).

## 11. What this notebook established

- PageRank at alpha 0.85 to tol 1e-12 on the primary 80,510-node graph
  converges in the printed iteration count and the vector sums to 1.
- The top-20 table, the clique sanity check, the PageRank distribution,
  the PageRank-vs-in-degree comparison (overlap, tau, scatter,
  disagreements) and the two sensitivities are all written to `results/`.
- The in-degree comparison (requirement 4) is quantified and the
  disagreements are explained by a verified structural fact.
- The dangling treatment (requirement 2) is correct and, on the primary
  graph, inactive; the peering-none sensitivity shows it working.

In [ ]:
print(f'notebook wall time so far: {time.perf_counter() - t_start:.1f} s')